# Generating Batch Inputs

Start with `protect(..., run=False)` to generate a complete pipeline
from a DataFrame, inspect its files, then protect multiple tables.
This API is new on the development branch; build the current source.

pytauargus can also **write** the batch input files, as a port of
[rtauargus](https://github.com/InseeFrLab/rtauargus) (the R wrapper).
The generators emit exactly the file formats the engine reads, so
you can script the full *input → batch → run* flow in pure Python.

| Function | Output | Ported from (R) |
|----------|--------|-----------------|
| `protect(…, run=False)` | `.asc` + `.rda` + `.arb` and planned `.tab` paths | Python pipeline |
| `micro_asc_rda(…)` | fixed-width `.asc` + `.rda` | `micro_asc_rda.R` |
| `micro_arb(…)` | `.arb` batch file | `micro_arb.R::micro_arb` |
| `write_rda` / `rda_text` | `.rda` metadata text | `micro_asc_rda.R::write_rda` |
| `write_hrc(…)` | `.hrc` hierarchy file | `hrc.R::write_hrc` |

`protect()` coordinates these generators and runs the engine when
`run=True` (the default). Use individual writers for custom batches.

## Setup

In [1]:
from pathlib import Path
import pytauargus

# Locate the repo's data/ fixtures (works from notebooks/ or repo root)
_here = Path.cwd()
DATA = next(
    (p / "data") for p in [_here, *_here.parents]
    if (p / "data" / "tau_testW.asc").exists()
)
OUT = _here / "out"          # scratch dir for notebook outputs (gitignored)
OUT.mkdir(exist_ok=True)
print(f"pytauargus {pytauargus.__version__}")
print("data : <repo>/data")
print("out  : ./out")

pytauargus 0.2.1
data : <repo>/data
out  : ./out


## 1. Preview the pipeline without running the engine

A flat list of explanatory variables specifies one table; a nested
list specifies several. `run=False` writes inputs and returns planned
output paths, with empty `.tables` and `.engine=None`.

In [2]:
import pandas as pd
from pytauargus import protect

# Twelve synthetic respondents; North/Retail has one dominant contributor.
micro = pd.DataFrame({
    "Region": ["North"] * 6 + ["South"] * 6,
    "Sector": ["Retail"] * 3 + ["Services"] * 3
              + ["Retail"] * 3 + ["Services"] * 3,
    "Turnover": [100., 5., 5., 10., 10., 10., 20., 20., 20., 30., 30., 30.],
})
micro

,Region,Sector,Turnover
0,North,Retail,100.0
1,North,Retail,5.0
2,North,Retail,5.0
3,North,Services,10.0
4,North,Services,10.0
5,North,Services,10.0
6,South,Retail,20.0
7,South,Retail,20.0
8,South,Retail,20.0
9,South,Services,30.0


In [3]:
tables = [["Region", "Sector"], ["Region"]]
draft = protect(
    micro, tables, response="Turnover", safety_rules="NK(2,75)",
    suppress="OPT(1)", run=False, workdir=str(OUT / "03_draft"),
)
assert draft.tables == [] and draft.engine is None
assert all(Path(draft.files[key]).exists() for key in ("asc", "rda", "arb"))
print("files:", {key: Path(path).name for key, path in draft.files.items()})
print("\n".join(Path(draft.files["rda"]).read_text().splitlines()[:16]))

files: {'asc': 'micro.asc', 'rda': 'micro.rda', 'arb': 'batch.arb', 'tab1': 'tab1.tab', 'tab2': 'tab2.tab'}
Region 1 5
  <RECODEABLE>
  <TOTCODE> "Total"
Sector 7 8
  <RECODEABLE>
  <TOTCODE> "Total"
Turnover 16 3
  <NUMERIC>
  <DECIMALS> 0


### Inspect the generated batch

The response and safety rule are recycled across tables. A single
suppression specification gets the correct table number (`OPT(1)`,
then `OPT(2)`). The batch uses type-5 `SO+` output for `TableResult`.

In [4]:
# Display relative paths so this preview is portable.
batch_text = Path(draft.files["arb"]).read_text()
print(batch_text.replace(str(Path(draft.workdir)), "<workdir>"))

// Batch generated by package *rtauargus*
// (2026-10-10 20:18:11 CEST)
<OPENMICRODATA> "<workdir>/micro.asc"
<OPENMETADATA> "<workdir>/micro.rda"
<SPECIFYTABLE> "Region""Sector"|"Turnover"||
<SAFETYRULE> NK(2,75)
<SPECIFYTABLE> "Region"|"Turnover"||
<SAFETYRULE> NK(2,75)
<READMICRODATA>
<SUPPRESS> OPT(1)
<WRITETABLE> (1,5,SO+,"<workdir>/tab1.tab")
<SUPPRESS> OPT(2)
<WRITETABLE> (2,5,SO+,"<workdir>/tab2.tab")




## 2. Run the multi-table pipeline

The default `run=True` executes the generated batch and returns
one `TableResult` per table. Use a separate directory to retain both
the file-only preview and the executed run.

In [5]:
from collections import Counter

protected = protect(
    micro, tables, response="Turnover", safety_rules="NK(2,75)",
    suppress="OPT(1)", workdir=str(OUT / "03_protect"),
)
assert [table.n_cells for table in protected.tables] == [9, 3]
for i, table in enumerate(protected.tables, 1):
    print(f"table {i}: {table.n_cells} cells; {dict(Counter(table.status()))}")
region = protected.tables[1]
region.dataframe()[["Region"]].assign(Turnover=region.safe())

table 1: 9 cells; {'S': 5, 'U': 2, 'M': 2}
table 2: 3 cells; {'S': 1, 'U': 1, 'M': 1}


,Region,Turnover
0,Total,290.0
1,North,x
2,South,x


## 3. `micro_arb` — write a custom `.arb` batch file

The main generator. Signature:

```python
micro_arb(
    arb_filename: str,          # output .arb path (default: temp file)
    asc_filename: str,          # input .asc (referenced, not created)
    rda_filename: str | None,  # defaults to asc_filename with .rda
    explanatory_vars: list,    # ["A","B"] = 1 table; [["A","B"],["C"]] = 2
    response_var: str | list,  # response variable name(s)
    safety_rules: str | list,  # e.g. "NK(1,85)"; recycled if shorter
    suppress: str | list,      # e.g. "OPT(.,0)" — "." = table number
    output_names: list,        # one per table
) -> {"arb_filename": str, "output_names": list}
```

In [6]:
from pytauargus.arb import micro_arb

res = micro_arb(
    arb_filename=str(OUT / "demo.arb"),
    asc_filename=str(DATA / "tau_testW.asc"),
    rda_filename=str(DATA / "tau_testW.rda"),
    explanatory_vars=[
        ["Size", "Region"],         # table 1
        ["Region", "IndustryCode"], # table 2
    ],
    response_var="Var2",
    safety_rules=["NK(2,75)"],     # recycled across both tables
    suppress="OPT(.,0)",           # "." -> table number; 0 = no time limit
    output_names=["out1.csv", "out2.csv"],
    output_type="1",             # CSV (the generator default is SBS/type 4)
)
print(res)
print()
print("=== generated demo.arb ===")
print((OUT / "demo.arb").read_text())

{'arb_filename': '/Users/trygve/Documents/Codex/2026-10-10/be/work/tauargus-checkout/bindings/python/notebooks/out/demo.arb', 'output_names': ['out1.csv', 'out2.csv']}

=== generated demo.arb ===
// Batch generated by package *rtauargus*
// (2026-10-10 20:18:11 CEST)
<OPENMICRODATA> "/Users/trygve/Documents/Codex/2026-10-10/be/work/tauargus-checkout/data/tau_testW.asc"
<OPENMETADATA> "/Users/trygve/Documents/Codex/2026-10-10/be/work/tauargus-checkout/data/tau_testW.rda"
<SPECIFYTABLE> "Size""Region"|"Var2"||
<SAFETYRULE> NK(2,75)
<SPECIFYTABLE> "Region""IndustryCode"|"Var2"||
<SAFETYRULE> NK(2,75)
<READMICRODATA>
<SUPPRESS> OPT(1,0)
<WRITETABLE> (1,1,,"out1.csv")
<SUPPRESS> OPT(2,0)
<WRITETABLE> (2,1,,"out2.csv")




### Per-table parameters

`safety_rules`, `suppress`, `output_type`, … all accept a single
value (recycled) or a per-table list:

In [7]:
res2 = micro_arb(
    arb_filename=str(OUT / "demo2.arb"),
    asc_filename=str(DATA / "tau_testW.asc"),
    rda_filename=str(DATA / "tau_testW.rda"),
    explanatory_vars=[["Size"], ["Region"]],
    response_var="Var2",
    safety_rules=["NK(1,85)", "FREQ(3,10)"],   # different rule per table
    suppress="MOD(.,0)",
    output_names=["s1.csv", "s2.csv"],
)
print((OUT / "demo2.arb").read_text())

// Batch generated by package *rtauargus*
// (2026-10-10 20:18:11 CEST)
<OPENMICRODATA> "/Users/trygve/Documents/Codex/2026-10-10/be/work/tauargus-checkout/data/tau_testW.asc"
<OPENMETADATA> "/Users/trygve/Documents/Codex/2026-10-10/be/work/tauargus-checkout/data/tau_testW.rda"
<SPECIFYTABLE> "Size"|"Var2"||
<SAFETYRULE> NK(1,85)
<SPECIFYTABLE> "Region"|"Var2"||
<SAFETYRULE> FREQ(3,10)
<READMICRODATA>
<SUPPRESS> MOD(1,0)
<WRITETABLE> (1,4,,"s1.csv")
<SUPPRESS> MOD(2,0)
<WRITETABLE> (2,4,,"s2.csv")




## 4. `write_rda` / `rda_text` — write `.rda` metadata

Each variable is a dict describing its position, width, type, and
optional codelist / hierarchy:

| Key | Required | Meaning |
|-----|----------|---------|
| `colname` | yes | Variable name |
| `position` | yes | 1-based byte position |
| `width` | yes | Field width |
| `type_var` | yes | `NUMERIC`, `RECODEABLE`, `WEIGHT`, `HOLDING` |
| `digits` | yes* | Decimal places (*NUMERIC / WEIGHT only) |
| `missing` | no | Missing-value code (``/`None` = none) |
| `totcode` | no | Total-code string |
| `codelist` | no | `.cdl` path (normalised to absolute) |
| `hierarchical` | no | `.hrc` file, level string like `2 1`, or `None` |
| `hierleadstring` | no | Hierarchy lead string (`.hrc` files) |

`write_rda` returns one block string per variable; `rda_text`
renders the whole file.

In [8]:
from pytauargus.rda import write_rda, rda_text

info_vars = [
    dict(type_var="RECODEABLE", colname="V1", position=1, width=1,
         digits=0, missing="?", totcode="Total", codelist=None,
         hierarchical=None, hierleadstring=None),
    dict(type_var="RECODEABLE", colname="V2", position=3, width=1,
         digits=0, missing="", totcode="Total", codelist=None,
         hierarchical="V2.hrc", hierleadstring="@"),
    dict(type_var="NUMERIC", colname="VAL", position=5, width=3,
         digits=2, missing="9999", totcode=None, codelist=None,
         hierarchical=None, hierleadstring=None),
    dict(type_var="WEIGHT", colname="POIDS", position=9, width=4,
         digits=1, missing="#", totcode=None, codelist=None,
         hierarchical=None, hierleadstring=None),
]

blocks = write_rda(info_vars)
print(f"{len(blocks)} variable blocks")
for b in blocks:
    print(repr(b))

print()
print("=== full .rda text ===")
print(rda_text(info_vars))

4 variable blocks
'V1 1 1 ?\n  <RECODEABLE>\n  <TOTCODE> "Total"'
'V2 3 1\n  <RECODEABLE>\n  <TOTCODE> "Total"\n  <HIERARCHICAL>\n  <HIERCODELIST> "V2.hrc"\n  <HIERLEADSTRING> "@"'
'VAL 5 3 9999\n  <NUMERIC>\n  <DECIMALS> 2'
'POIDS 9 4 #\n  <WEIGHT>\n  <DECIMALS> 1'

=== full .rda text ===
V1 1 1 ?
  <RECODEABLE>
  <TOTCODE> "Total"
V2 3 1
  <RECODEABLE>
  <TOTCODE> "Total"
  <HIERARCHICAL>
  <HIERCODELIST> "V2.hrc"
  <HIERLEADSTRING> "@"
VAL 5 3 9999
  <NUMERIC>
  <DECIMALS> 2
POIDS 9 4 #
  <WEIGHT>
  <DECIMALS> 1



## 5. `write_hrc` — write a `.hrc` hierarchy file

Builds a hierarchy from microdata columns ordered **finest →
coarsest** and writes it as a `.hrc` file (child levels are
prefixed with the lead string, default `@`).

```python
write_hrc(
    microdata: dict,      # {var: [value, …]}  (values: str | None)
    vars_hrc: str | list, # column name(s), finest → coarsest
    hierleadstring: str = "@",
    hrc_filename: str | None,   # default: temp file
    fill_na: str = "up",        # "up"/"down" NA imputation
    compact: bool = True,       # prune single-value branches
    hierlevels: str | None,     # split one code into positional levels
) -> str  # the written path
```

In [9]:
from pytauargus.hrc import write_hrc
import warnings

# A small hierarchy: city (finest) → country → continent
microdata = {
    "city":      ["PAR", "LYO", "BER", "MUC", "NYC", "TOR", "TYO"],
    "country":   ["FR",  "FR",  "DE",  "DE",  "US",  "CA",  "JP"],
    "continent": ["EU",  "EU",  "EU",  "EU",  "NA",  "NA",  "AS"],
}

with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # silence single-level/NA warnings in demos
    hrc_path = write_hrc(
        microdata,
        vars_hrc=["city", "country", "continent"],
        hrc_filename=str(OUT / "geo.hrc"),
    )

print(f"wrote {hrc_path}")
print()
print((OUT / "geo.hrc").read_text())

wrote /Users/trygve/Documents/Codex/2026-10-10/be/work/tauargus-checkout/bindings/python/notebooks/out/geo.hrc

AS
@JP
@@TYO
EU
@DE
@@BER
@@MUC
@FR
@@LYO
@@PAR
NA
@CA
@@TOR
@US
@@NYC



### `hierlevels` — split one code into positional levels

When a single column encodes several levels positionally (e.g.
`"123"` = level 1 `1`, level 2 `2`, level 3 `3`), `hierlevels`
splits it — exactly one `vars_hrc` column is allowed:

In [10]:
import warnings

microdata2 = {"code": ["111", "112", "121", "122", "211", "212"]}

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    hrc2 = write_hrc(
        microdata2,
        vars_hrc="code",
        hierlevels="1 1 1",           # 3 levels, 1 character each
        hrc_filename=str(OUT / "pos.hrc"),
    )

print((OUT / "pos.hrc").read_text())

1
@11
@@111
@@112
@12
@@121
@@122
2
@21
@@211
@@212



## 6. Custom batch round-trip: generate → run → inspect

A generated `.arb` feeds straight back into the engine. This is
the full scripted pipeline — note the in-arb `<SUPPRESS> OPT(…)`
runs during `run_batch`:

In [11]:
from pytauargus.engine import run_batch

arb_path = OUT / "demo.arb"
print(f"Running {arb_path.name}…\n")

eng = run_batch(arb_path)
for i in range(eng._n_tables):
    ncell, _ = eng._tau.get_total_table_size(i)
    spec, _ = eng._tables[i]
    print(f"  table {i+1}: {' x '.join(spec.exp_vars):<24} | {spec.resp_var}  →  {ncell} cells")

Running demo.arb…



  table 1: Size x Region            | Var2  →  162 cells
  table 2: Region x IndustryCode    | Var2  →  12798 cells


### Where did the outputs go?

`WRITETABLE` output files are written relative to the **`.arb`
file's directory** — here, the `out/` scratch dir:

In [12]:
for p in sorted(OUT.iterdir()):
    if p.is_file():
        print(f"  {p.name:<16} {p.stat().st_size:>8} bytes")

  01_dataframe_publication.csv      185 bytes
  02_dataframe_audit.tab      558 bytes
  02_dataframe_publication.csv      185 bytes
  demo.arb              528 bytes
  demo2.arb             504 bytes
  geo.hrc                71 bytes
  out1.csv              824 bytes
  out2.csv            55602 bytes
  pos.hrc                52 bytes
  table1.csv            700 bytes
  table1.tab          10976 bytes


## Gotchas & known limitations

- `suppress="OPT(.,0)"` uses max-time `0` (no limit), so this example
  can finish without a deadline. Positive time limits are in minutes.
- `protect()` and `micro_asc_rda()` write the `.asc` file too;
  `micro_arb()` only references an existing `.asc` / `.rda` pair.
- Working directories and original values are retained for analysis;
  publish the masked `safe()` response, and clean up intermediates yourself.
- `write_hrc` with a **single** column emits a flat sorted list and
  warns; `fill_na` warns when NAs are imputed.

## Summary

You've now:

1. Previewed `.asc` / `.rda` / `.arb` with `protect(run=False)`
2. Protected multiple tables and read their `TableResult` objects
3. Generated custom `.arb` batch files with `micro_arb`
4. Generated `.rda` metadata text with `write_rda` / `rda_text`
5. Generated `.hrc` hierarchy files with `write_hrc`
6. Round-tripped: generated batch → engine → tables + CSVs

**Back to:** [01_quickstart.ipynb](01_quickstart.ipynb) ·
[02_protection_and_audit.ipynb](02_protection_and_audit.ipynb)